In [ ]:
import mlflow
import ipywidgets as widgets
from ipywidgets import interact

import os
import pandas as pd

In [ ]:
experiments = mlflow.search_experiments()
exp_id2name = { experiment.name: experiment.experiment_id for experiment in  experiments }

In [ ]:
experiments = widgets.SelectMultiple(
    options=exp_id2name,
    description="Experiments"
)

# Botón para ejecutar
button = widgets.Button(description="Buscar runs")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        runs_df = mlflow.search_runs(experiment_ids=list(experiments.value))
        runs_df = runs_df.query("`metrics.best_val_loss` < 12")
        runs_df = runs_df.drop(["artifact_uri", "status", "experiment_id", "start_time", "end_time"], axis=1)
        # runs_df = runs_df.drop(["start_time", "end_time"], axis=1)
        display(runs_df)

button.on_click(on_button_clicked)

# Mostrar widgets
display(widgets.VBox([experiments, button, output]))

In [ ]:
dir()

In [ ]:
run = mlflow.get_run("8fbd48be135e4391bbb655b5c2f914b9")

In [ ]:
AUCDIR = f"{run.info.artifact_uri}/auc/"
unpooled_auc = pd.read_parquet(f"{AUCDIR}/df_auc_unpooled.parquet")
both_auc = pd.read_parquet(f"{AUCDIR}/df_both.parquet")

In [ ]:
unpooled_auc = unpooled_auc.query("n_diseased > 100").drop(["auc"], axis=1)
unpooled_auc = unpooled_auc[~unpooled_auc.duplicated()]
unpooled_auc